<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_5_Ensemble/18_5_0_Intro_Ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ensemble Methods: A Deeper Dive

**Author:** Brad Sheese

---

## Introduction: Building on What We Know

In the 18_1 series, we were introduced to tree-based methods: decision trees, random forests, gradient boosting, and XGBoost. We saw how they outperformed linear models on the Ames Housing dataset and learned about concepts like overfitting, feature importance, and the bias-variance tradeoff.

This series (18_5) goes **deeper**. We will:

1.  **Examine each ensemble method in isolation** — understanding exactly *how* bagging, random forests, and boosting work under the hood.
2.  **Focus on classification-specific concerns** — precision, recall, false negatives in medical diagnosis, and ROC curves.
3.  **Tune hyperparameters systematically** — using GridSearchCV to find optimal settings rather than guessing.
4.  **Compare all five methods head-to-head** — single tree, bagging, random forest, boosting, and BART — using nested cross-validation for unbiased estimates.

We'll use the **Wisconsin Breast Cancer dataset** — a medical diagnosis problem where the cost of a false negative (missing a cancer) is far higher than a false positive (unnecessary biopsy). This makes it an ideal test case for understanding the precision-recall tradeoff in real-world applications.

### Recommended Preparation
- [StatQuest: Random Forests](https://www.youtube.com/watch?v=J4Wdy0Wc_xQ&t=143s) — excellent visual explanation of the core concepts
- Review the 18_1_5 notebook on tree-based methods if you need a refresher

## The Ensemble Hierarchy

All ensemble methods share the same core idea: **combine multiple weak models to create a strong one**. But they differ in *how* they build and combine those models.

| Method | How Trees Are Built | How Predictions Are Combined | Primary Goal |
|---|---|---|---|
| **Single Tree** | One tree, grown top-down | N/A | Interpretability |
| **Bagging** | Independent trees on bootstrap samples | Majority vote / average | Reduce variance |
| **Random Forest** | Independent trees + random feature subsets | Majority vote / average | Reduce variance + decorrelate |
| **Boosting** | Sequential trees, each correcting errors | Weighted sum | Reduce bias |
| **BART** | Bayesian tree perturbation | Posterior distribution | Reduce both + quantify uncertainty |

### Key Insight: Variance vs. Bias

- **Bagging and Random Forests** reduce **variance** — they make unstable models (deep trees) more stable by averaging many of them.
- **Boosting** reduces **bias** — it takes weak models (shallow trees) and makes them stronger by sequentially correcting their mistakes.
- **BART** addresses **both** while also providing uncertainty estimates (credible intervals).

In the notebooks that follow, we'll see each of these in action.

## The Dataset: Wisconsin Breast Cancer

We'll use the Wisconsin Diagnostic Breast Cancer (WDBC) dataset throughout this series. It contains measurements from digitized images of fine needle aspirates of breast masses.

- **569 samples**, **30 features** (mean, standard error, and "worst" values for 10 cell nucleus characteristics)
- **Binary target**: Malignant (1) or Benign (0)
- **Class distribution**: ~63% benign, ~37% malignant — moderately imbalanced

This is a **high-stakes classification problem**. A false negative (predicting benign when the tumor is malignant) means a cancer goes undiagnosed. A false positive (predicting malignant when the tumor is benign) means an unnecessary biopsy. These costs are not equal — and that's exactly why we need to look beyond accuracy.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df = data.frame

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['target'].value_counts())
print(f"\nClass proportions:")
print(df['target'].value_counts(normalize=True))
print(f"\nFeatures: {len(data.feature_names)}")
print(f"Target names: {dict(enumerate(data.target_names))}")

Dataset shape: (569, 31)

Class distribution:
target
1    357
0    212
Name: count, dtype: int64

Class proportions:
target
1    0.627417
0    0.372583
Name: proportion, dtype: float64

Features: 30
Target names: {0: np.str_('malignant'), 1: np.str_('benign')}


### Why This Dataset Matters

The 37% malignant rate means a naive model that always predicts "benign" would achieve 63% accuracy. Any useful model must significantly exceed this baseline.

More importantly, **accuracy alone is insufficient**. If a model achieves 95% accuracy but misses 30% of actual cancers, it's clinically useless. We need to evaluate precision, recall, and the ROC curve — metrics we covered in 18_1_1 and 18_1_2.

In the next notebook, we'll start with the building block: a single decision tree.